* AI was used to ask questions realted to tsne

In [1]:
import pandas as pd

df = pd.read_csv("../../Datasets/cleaned_adult_data.csv")  
df = df.drop_duplicates()

In [2]:
'''
REQUIREMENTS
All columns MUST be numeric. 
`max_iter` minimum is 250.

FUNCTION:
This function splits the target with it's features, scales the features (standard scaler), and creates
a t-SNE plot with a slider using the given max iterations. 

PARAMETERS
df            : Pandas dataframe
target_column : string (of pandas series/column)
max_iter      : Number of adjustments to initial arrangement of points.

HOW TO READ t-SNE
* Are the clusters clearly seperated? (Then start with a simple ML model)
* Are the clusters overlapping/noisy? (Understand why overlap exists/Clean data/Complex model)
* Are there any isolated points? (Suggests better outlier mitigation)
* Are there any non linear patterns? (Then try nonlinear models, RandomForest, Gradient Boost, ANN)
'''
def plot_tsne(df, target_column, max_iter=1000):

    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import ipywidgets as widgets
    from IPython.display import display
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler

    # Set the theme to dark to avoid eye-burning sensations.
    sns.set_theme(style="dark")

    # Separate features and target
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    # Scale the features
    X_scaled = StandardScaler().fit_transform(X)

    # Function called when the slider changes
    def update_tsne(perplexity):

        tsne = TSNE(
            n_components=2,
            perplexity=perplexity,
            max_iter=max_iter,
            init="pca",
            random_state=42,
            learning_rate="auto"
        )

        embedded = tsne.fit_transform(X_scaled)

        # Create DataFrame for Seaborn
        tsne_df = pd.DataFrame({
            "TSNE_1": embedded[:, 0],
            "TSNE_2": embedded[:, 1],
            "target": y.to_numpy()
        })

        # IF YOU ARE NOT SATISFIED WITH THE SIZE
        # CHANGE IT HERE (X, Y)
        plt.figure(figsize=(13, 7))

        # Now, we make the scatterplot -- Using T-SNE dimensions to plot!
        sns.scatterplot(
            data=tsne_df,
            x="TSNE_1",
            y="TSNE_2",
            hue="target",
            palette="viridis"
        )

        plt.title(f" t-SNE, Iterations: {max_iter}")
        plt.show()

    # Finally, we create a slider to manage the perplexity.
    slider = widgets.IntSlider(
        value=30,
        min=5,
        max=min(50, len(df) - 1),
        step=1,
        description="Perplexity:",
        continuous_update=False
    )

    display(widgets.interactive(update_tsne, perplexity=slider))

In [3]:
# The functionn needs the data to be in numeric format, so using clean csv
# tsne is very slow, so we will sample the data to 3000 rows
df_tsne = df.sample(3000, random_state=42).copy()
# the target column is categorical, so we need to convert it to numeric
df_tsne["income"] = (df_tsne["income"] == ">50K").astype(int)

In [4]:
plot_tsne(df_tsne, "income", max_iter=1000)

interactive(children=(IntSlider(value=30, continuous_update=False, description='Perplexity:', max=50, min=5), …

* At perplexity 5 the plot gives many small, fragmemted clumps, and at 50 it gives larger broader groups.

### Let's change the max_iter

In [5]:
plot_tsne(df_tsne, "income", max_iter=250)

interactive(children=(IntSlider(value=30, continuous_update=False, description='Perplexity:', max=50, min=5), …

In [6]:
plot_tsne(df_tsne, "income", max_iter=2000)

interactive(children=(IntSlider(value=30, continuous_update=False, description='Perplexity:', max=50, min=5), …

* With 250 iterations the plot was not yet settled. With 1000 and 2000 it looked almost the same, so the embedding has stabilized by 1000 iterrations.

* How complex are the relationships of your features according to t-SNE?

--> There are many small groups visible with a mix of both income colors, so not a simple pattern.

* Are you finding any isolated cases?

--> There are obviously few points that are far from others, but not that many.

* Does any overlap exist?

--> Yes, most groups contains both the colors, although a few groups have one class more

* If you were to train a model with this dataset, which would you try first? (hypothetically, of course!)

--> Because there is heavy overlap, I would use the stronger model such as Random Forest or GradientBoost (as suggested in prof's notebook).
